In [6]:
import json
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from scipy.signal import lfilter
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, roc_auc_score

from common import Featurizer, load_manifest, _expanding_z, _welford_check


FRAME = 3
STAGE0_TAG = 'mnq-TICK-9-12am'

MANIFEST_PATH = f"stage-0/manifest_{STAGE0_TAG}_{FRAME}s.json"
BARS_PATH = f"stage-0/bars_{STAGE0_TAG}_{FRAME}s.pqt"
EVENTS_PATH = f"stage-0/events_{STAGE0_TAG}_{FRAME}s.pqt"

ITER_DIR = "."                                   # iteration-2 root (encapsulated)
OUT_DIR = "stage-5"

VALID_FROM = "2025-07-01"
TRAIN_END = "2025-12-31"
TEST_FROM = "2026-01-01"

# frozen 5c2 architecture constants (NOT fork axes -- stay in code)
TAUS = (2.0, 6.0, 18.0)
VALUE_LAGS = (1, 2)                               # t-2, t-3 (t-1 shift is implicit)
ZWARM = 20
TOD_BIN_MIN = 30
BODY_OPEN_COL = "Open"
BODY_CLOSE_COL = "Last"
BODY_TAG = "ha"

LGBM_PARAMS = dict(
    objective="binary", metric="binary_logloss", learning_rate=0.05,
    num_leaves=127, min_data_in_leaf=1000, feature_fraction=0.9,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    num_threads=16, verbosity=-1,
)
NUM_ROUNDS = 8000
EARLY_STOP = 200
STAGE5_ANCHOR = 0.27402

#bars = pd.read_parquet(BARS_PATH)
#events = pd.read_parquet(EVENTS_PATH)
source_file = "data/mnq-tick-full-3sec.pqt"
src = pd.read_parquet(source_file)


In [3]:
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [ ]:
#print(bars.head())
#print(events.head())
print(src.info())
print(src.head(50))